## Load the Last.fm Dataset

In [45]:
from implicit.datasets.lastfm import get_lastfm

artists, users, artist_user_plays = get_lastfm()

## Peek Inside the Last.fm Data

In [50]:
print(f"Artists: {len(artists):,}")
print(f"User: {len(users):,}")
print(f"Matrix shape (artists x users): {artist_user_plays.shape}")
print(f"Non-zero interactions: {artist_user_plays.nnz:,}")

Artists: 292,385
User: 358,868
Matrix shape (artists x users): (292385, 358868)
Non-zero interactions: 17,535,606


## Train-Test Split

In [47]:
from implicit.evaluation import train_test_split

# Transpose to (users x artists) format | rows -> users | columns -> items
user_plays = artist_user_plays.T.tocsr()

train_raw, test_raw = train_test_split(user_plays,
                                       train_percentage=0.8,
                                       random_state=42)

print(f"Train shape: {train_raw.shape}, non-zero: {train_raw.nnz:,}")
print(f"Test shape: {test_raw.shape}, non-zero: {test_raw.nnz:,}")

Train shape: (358868, 292385), non-zero: 14,028,047
Test shape: (358868, 292385), non-zero: 3,507,558


## Preprocess Data for Training

- **`bm25_weight`**: Applies BM25 scoring to user-item interactions, reducing the influence of popular items.

- **`csr_matrix`**: A sparse matrix format (Compressed Sparse Row) that stores only non-zero values efficiently.

```python
# Dense matrix (wastes space)
dense = [
    [0, 0, 5],
    [3, 0, 0],
    [0, 2, 0]
]

# CSR format (stores only non-zero)
from scipy.sparse import csr_matrix
sparse = csr_matrix(dense)

print(sparse.data)    # [5, 3, 2] - values
print(sparse.indices) # [2, 0, 1] - columns  
print(sparse.indptr)  # [0, 1, 2, 3] - row starts

## BM25 Weighting for ALS

In [67]:
from implicit.nearest_neighbours import bm25_weight

# ALS: apply BM25 weighting on train_raw
weighted_train_als = bm25_weight(train_raw,     # Sparse matrix (artists x users) with play counts
                                 K1=100,     # Saturation: higher = more weight for repeated plays
                                 B=0.8).tocsr()      # Normalization: higher = penalizes very active users more)

test_als = test_raw.sign()

## Train ALS Model

**Confidence Weight**  
`Confidence = 1 + α × (interaction strength)`

Higher confidence = more important to the model.  
Common `α` values: 1.0 to 40.0.

| Platform | Action | Weight |
|----------|--------|--------|
| E-commerce | Purchase | 10.0 |
| E-commerce | Add to cart | 5.0 |
| E-commerce | Click | 1.0 |
| E-commerce | View | 0.5 |
| YouTube | Watch time (minutes) | 1.0 - 5.0 (scaled) |
| YouTube | Like | 10.0 |
| YouTube | Subscribe | 20.0 |

In [73]:
from implicit.als import AlternatingLeastSquares

# Initialize ALS model
als_model = AlternatingLeastSquares(factors=128,                  # Latent factor dimension (embedding size)
                                    regularization=0.01,          # Prevents overfitting
                                    alpha=20.0,                   # confidence weight for positive interactions (plays/listens)
                                    iterations=50)                # Training epochs


# Train the model
als_model.fit(user_items=weighted_train_als,
              show_progress=True)

100%|██████████| 50/50 [02:29<00:00,  2.99s/it]


## Binary Conversion for BPR and LMF

In [55]:
# converts non-zero to 1
train_binary = train_raw.sign()
test_binary = test_raw.sign()

## Train BPR Model

In [56]:
from implicit.bpr import BayesianPersonalizedRanking

bpr_model = BayesianPersonalizedRanking(factors=100,
                                        learning_rate=0.01,
                                        regularization=0.01,
                                        iterations=100,
                                        verify_negative_samples=True)

bpr_model.fit(user_items=train_binary,
              show_progress=True)

100%|██████████| 100/100 [03:48<00:00,  2.28s/it, train_auc=95.25%, skipped=1.75%]


## Train LMF Model

In [63]:
from implicit.lmf import LogisticMatrixFactorization

lmf_model = LogisticMatrixFactorization(factors=30,
                                        learning_rate=1.0,
                                        regularization=0.01,
                                        iterations=30)

lmf_model.fit(user_items=train_binary,
              show_progress=True)

100%|██████████| 30/30 [00:43<00:00,  1.46s/it]


## Models Evaluation

### 🎯 Binary Evaluation (Hits only)

| | |
|---|---|
| **Question** | "Did the user interact with this item at all?" |
| **Best for** | Discovery, new user onboarding, CTR optimization |
| **Fair to** | ✅ BPR / LMF (native format) <br> ⚠️ ALS (slightly handicapped) |

---

### ⚡ Weighted Evaluation (Engagement strength)

| | |
|---|---|
| **Question** | "Did the user strongly engage with this item?" |
| **Best for** | Retention, watch time, play-count-aware recommendations |
| **Fair to** | ✅ ALS (native format) <br> ❌ BPR / LMF (heavily handicapped) |

---

### 📈 Quick Summary

| Strategy | Best For | Fair To |
|----------|----------|---------|
| **Binary** | Discovery, CTR | BPR / LMF |
| **Weighted** | Retention, Engagement | ALS |

---

### 🎯 Final Takeaway

> **For predicting *whether* a user will listen (discovery) → BPR works best.**  
> **For predicting *how much* they'll listen (engagement) → ALS is the right choice.**

**Different questions. Different models. Both valid.**

### Evaluating ALS Model


In [74]:
from implicit.evaluation import ranking_metrics_at_k

als_metrics = ranking_metrics_at_k(als_model, weighted_train_als, test_als, K=10)

100%|██████████| 358511/358511 [04:05<00:00, 1462.61it/s]


### Evaluating BPR Model

In [69]:
bpr_metrics = ranking_metrics_at_k(bpr_model, train_binary, test_binary, K=10)

100%|██████████| 358511/358511 [03:58<00:00, 1500.31it/s]


### Evaluating LMF Model

In [70]:
lmf_metrics = ranking_metrics_at_k(lmf_model, train_binary, test_binary, K=10)

100%|██████████| 358511/358511 [03:48<00:00, 1570.54it/s]


## Model Performance Comparison

### Metrics Explained

| Metric | Question it answers | Example |
|--------|---------------------|---------|
| **Precision@10** | "Out of 10 recs, how many did user actually like?" | 0.054 = 5.4% → ~5 good recs out of 100 (sparse dataset) |
| **MAP** | "Are the good recs at the TOP of the list?" | 0.02 = low but expected for Last.fm |
| **NDCG** | "Does ranking order make sense? (valuing top positions more)" | 0.05 = model learns something, not random |
| **AUC** | "Given one liked + one disliked, does model pick the liked one?" | 0.50 = coin flip. 0.52 = barely better (normal for sparse data) |


**Why ALS wins for Last.fm:**

| Aspect | ALS | BPR | LMF |
|--------|:---:|:---:|:---:|
| Precision | 5.4% | 6.4% | 3.3% |
| **Play counts matter?** | ✅ Yes | ❌ No | ❌ No |
| **Fair eval on weighted** | ✅ | ❌ | ❌ |

**Final verdict:**
> BPR has higher precision, but ALS answers the **right question** for Last.fm:  
> *"How much will they listen?"* not *"Will they listen at all?"*

In [131]:
import pandas as pd

df = pd.DataFrame([als_metrics, bpr_metrics, lmf_metrics], index=['ALS', 'BPR', 'LMF'])

df.round(4)

,precision,map,ndcg,auc
ALS,0.0544,0.0206,0.0534,0.5243
BPR,0.0640,0.0276,0.0673,0.5286
LMF,0.0327,0.0122,0.0325,0.5145


## Examine Learned Factors


**What the numbers mean:**

- Each user has **128 numbers** representing their "taste vector"
- Each artist has **128 numbers** representing their "vibe vector"
- **Similar taste vectors × similar vibe vectors = high recommendation score**

In [85]:
print(f"User factors shape: {als_model.user_factors.shape}")  # (n_users, factors)
print(f"Item factors shape: {als_model.item_factors.shape}")  # (n_items, factors)

User factors shape: (358868, 128)
Item factors shape: (292385, 128)


In [132]:
# Embedding vector for first user
als_model.user_factors[1]

array([ -3.0461802 ,  -2.8639257 ,  -6.3303075 , -11.920091  ,
        13.269592  ,  -5.0577927 ,   4.528689  ,  -8.872518  ,
         4.5552683 ,   0.20538217, -10.741179  , -23.124702  ,
         5.8795285 ,   2.6789672 ,  11.122774  ,   2.1950128 ,
        -6.2314353 ,  -0.6678439 ,   9.181217  ,   1.6355008 ,
       -10.173349  , -10.557771  ,   5.925866  , -17.2074    ,
         4.6810055 ,   5.282388  ,   6.76831   ,  -7.1360903 ,
        -1.5489788 ,   1.4601886 ,   1.5752914 ,  -0.27815226,
        -1.7600963 ,  22.128426  ,  -9.803713  ,  -1.9886767 ,
        -9.005264  ,  -2.0631115 ,   9.905556  ,  12.061962  ,
        -3.4855375 ,  -1.2129947 ,   7.0654874 ,   1.439276  ,
        10.468531  ,   8.212989  ,   8.007538  ,   2.3814    ,
        -7.3255067 ,   5.6006737 ,   4.941792  ,  -5.2201686 ,
       -13.56673   ,   1.9725558 ,  -2.129661  ,  -5.790408  ,
         7.26275   ,  14.89273   ,   4.625127  , -13.324672  ,
        17.601952  , -10.944936  ,  -2.988873  , -11.06

## Training Final ALS Model on Full Data
Now that we've validated the model works, we train on **100% of the data** for maximum performance.

In [128]:
from implicit.nearest_neighbours import bm25_weight
from implicit.als import AlternatingLeastSquares

user_plays = artist_user_plays.T.tocsr()

user_plays_weighted = bm25_weight(user_plays, K1=100, B=0.8).tocsr()

# Initialize ALS model
final_als_model = AlternatingLeastSquares(factors=128,           # Latent factor dimension (embedding size)
                                          regularization=0.01,   # Prevents overfitting
                                          alpha=20.0,            # confidence weight for positive interactions (plays/listens)
                                          iterations=50)         # Training epochs


# Train the model
final_als_model.fit(user_items=user_plays_weighted,
                    show_progress=True,)

100%|██████████| 50/50 [02:53<00:00,  3.47s/it]


## Make Recommendations

In [193]:
# Select a specific user (by index position)
user_id = 100

# Get user's play history (artists and weighted counts) - model automatically excludes these from recommendations
recs = final_als_model.recommend(userid=user_id,
                                 user_items=user_plays_weighted[user_id],
                                 N=10)

# The recommend() method returns TWO separate arrays
artist_ids, scores = recs

# Loop through both arrays simultaneously
for artist_id, score in zip(artist_ids, scores):    # zip pairs them together
    print(f"{artists[artist_id]}: {score:.4f}")

astrud gilberto: 1.7537
a-ha: 1.6146
the radio dept.: 1.5037
asobi seksu: 1.4898
bajofondo tango club: 1.4863
the last shadow puppets: 1.4279
black box recorder: 1.3548
astor piazzolla: 1.3542
the futureheads: 1.3184
junior boys: 1.3148


## Exploring a User's Liked Artists

In [194]:
# Select a specific user (by index position)
user_id = 100

# Get all artist IDs that this user has listened to (non-zero interactions)
liked_artists = user_plays[user_id].indices

# Loop through the first 20 artist IDs the user listened to
for artist_id in liked_artists[:20]:
    
    # Convert artist ID to actual name and print it
    print(artists[artist_id])

almodóvar, pedro
antonio carlos jobim & astrud gilberto
antony and the johnsons
beastie boys
beirut
benjamin biolay
best of
black eyed peas
christina rosenvinge
cocorosie
diariu
djavan
feist
flash cadillac & the continental kids
flight of the conchords
frank popp ensemble
franz ferdinand
gotan project
instituto mexicano del sonido
iván ferreiro


In [192]:
import numpy as np

# Explain why an artist was recommended
def explain_recommendation(user_id, artist_id, model, top_n=3):
    user_vec = model.user_factors[user_id]
    artist_vec = model.item_factors[artist_id]
    
    # Calculate similarity score
    score = np.dot(user_vec, artist_vec)
    
    # Find similar users (excluding self)
    similar_users = model.similar_users(user_id, N=top_n + 1)
    
    print(f"Recommended because:")
    print(f"  - Your taste vector × artist vibe vector = {score:.4f}")
    print(f"  - Similar to users who like {artists[artist_id]}")
    
    # Show similar users' top artists (skip first if it's the user itself)
    print(f"\n  Users with similar taste also listen to:")
    count = 0
    for item in similar_users:
        similar_user_id = int(item[0])
        similarity_score = float(item[1])
        
        # Skip the user itself
        if similar_user_id == user_id:
            continue
            
        if count >= top_n:
            break
            
        # Get top artist for that similar user
        similar_user_artists = user_plays[similar_user_id].indices
        if len(similar_user_artists) > 0:
            top_artist = artists[similar_user_artists[0]]
            print(f"    - User {similar_user_id} (similarity: {similarity_score:.3f}) likes {top_artist}")
            count += 1
    
    return score

# Test it
user_id = 326348
recommended_artist_id = 779  # Scorpions
score = explain_recommendation(user_id, recommended_artist_id, final_als_model)

Recommended because:
  - Your taste vector × artist vibe vector = -0.0200
  - Similar to users who like 021 eminem

  Users with similar taste also listen to:
    - User 1 (similarity: 0.619) likes ai aso


## Why ALS recommends Rock for Eminem fans

**ALS doesn't understand music genres.** It only finds patterns in listening behavior.

- **Eminem listeners** → also listen to **Incubus, The Killers** (rock bands)
- **Model assumption:** If users listen together, they're "similar"

### The Problem: Popularity Bias

Popular artists cluster together because **everyone** listens to them, regardless of genre. `(popularity bias)`

| You asked for | ALS gave you |
|--------------|--------------|
| Music that sounds like Rap | Music that Rap fans also listen to (Rock/Alternative) |

**ALS prioritizes co-occurrence ("fans also like") over audio features (genre).**

In [209]:
# Find artists similar to a given artist
artist_name = "eminem"
artist_id = list(artists).index(artist_name)
similar_ids, similar_scores = final_als_model.similar_items(artist_id, N=10)

for aid, score in zip(similar_ids, similar_scores):
    print(f"{artists[aid]}: {score:.4f}")

eminem: 1.0000
incubus: 0.9996
the killers: 0.9995
foo fighters: 0.9995
marilyn manson: 0.9994
guns n roses: 0.9994
kanye west: 0.9993
the kooks: 0.9993
green day: 0.9992
tool: 0.9992
